# Chapter 8: Support Vector Machines (SVM)
# Part 5 — SVM in Practice (scikit-learn & Production)

---

# Learning Objectives

By the end of this chapter, you will understand:

- SVC vs LinearSVC vs NuSVC
- Why feature scaling is critical
- Building SVM pipelines
- Hyperparameter tuning
- Cross Validation
- GridSearchCV
- Model Evaluation
- Probability Calibration
- Saving Models
- Production Best Practices
- Common Mistakes

---

# 1. SVM in scikit-learn

Scikit-learn provides three main SVM classifiers.

```
SVC

LinearSVC

NuSVC
```

Although they solve similar problems, they are designed for different situations.

---

# 2. SVC

```python
from sklearn.svm import SVC

model = SVC()
```

Uses

LIBSVM

internally.

Supports

- Linear Kernel
- Polynomial Kernel
- RBF
- Sigmoid
- Custom Kernels

---

Advantages

✔ Supports kernels

✔ Probability estimation

✔ Flexible

---

Disadvantages

Training becomes slow for

large datasets.

---

Recommended Dataset Size

```
100

↓

50,000 samples
```

---

# 3. LinearSVC

```python
from sklearn.svm import LinearSVC
```

Uses

LIBLINEAR

instead of LIBSVM.

Important

LinearSVC

does NOT support

non-linear kernels.

Only

Linear Hyperplanes.

---

Advantages

Very Fast

Handles

Millions

of features.

Perfect for

NLP

Spam Detection

Sentiment Analysis

Document Classification

---

Disadvantages

No Kernel Trick.

---

# 4. NuSVC

Rarely used.

Instead of parameter

C

it uses

ν (Nu)

which controls

Support Vectors

and

Training Errors.

Mostly used

in research.

---

# Comparison

| Feature | SVC | LinearSVC | NuSVC |
|----------|-----|-----------|--------|
| Linear Kernel | ✅ | ✅ | ✅ |
| Polynomial | ✅ | ❌ | ✅ |
| RBF | ✅ | ❌ | ✅ |
| Sigmoid | ✅ | ❌ | ✅ |
| Fast | ❌ | ✅ | ❌ |
| Large Datasets | ❌ | ✅ | ❌ |
| NLP | Good | Excellent | Rare |

---

# 5. Why Scaling is Necessary

This is one of the biggest mistakes beginners make.

Consider

Height

```
170 cm
```

Salary

```
80,000
```

Age

```
25
```

Without scaling,

Salary dominates.

Distance calculations become meaningless.

---

Remember

SVM depends heavily on

Distances

Dot Products

Margins

All of these are affected by feature scale.

---

# Example

Without Scaling

```
Feature A

0 - 1

Feature B

0 - 100000
```

The second feature dominates.

The hyperplane becomes biased.

---

# Solution

Always scale

numeric features.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
```

---

# Why Didn't We Scale TF-IDF?

In your Spam Detection project

you used

TF-IDF.

TF-IDF already normalizes

feature values.

Therefore

StandardScaler

is

NOT required.

This is one reason

LinearSVC

works beautifully for text.

---

# 6. Building a Pipeline

Never do

```python
X = scaler.fit_transform(X)

model.fit(X)
```

Instead

```python
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])
```

Advantages

✔ Cleaner

✔ Prevents Data Leakage

✔ Easy Deployment

✔ GridSearchCV compatible

---

# Example

```python
pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
```

One line.

Everything happens automatically.

---

# 7. Why Pipelines Matter

Imagine

You forget

to scale

new data.

Prediction becomes wrong.

Pipeline guarantees

the same preprocessing

during training

and

prediction.

This is why

production systems

always use pipelines.

---

# 8. Hyperparameters

Most important parameters.

---

## kernel

```python
kernel="linear"
```

Linear boundary.

---

```python
kernel="rbf"
```

Non-linear boundary.

---

## C

Controls

Margin

vs

Classification Errors.

Small

↓

Simple Model.

Large

↓

Complex Model.

---

## gamma

Only

RBF

Polynomial

Sigmoid.

Controls

Influence Radius.

---

## degree

Only

Polynomial Kernel.

Higher

↓

More Complex Boundary.

---

## probability

```python
probability=True
```

Enables

predict_proba().

Costs

additional training time.

---

# 9. GridSearchCV

Instead of guessing

hyperparameters

let

GridSearchCV

search them.

Example

```python
from sklearn.model_selection import GridSearchCV

params = {

    "C":[0.1,1,10],

    "kernel":["linear","rbf"],

    "gamma":["scale","auto"]
}

grid = GridSearchCV(

    SVC(),

    params,

    cv=5,

    scoring="f1"

)

grid.fit(X_train,y_train)
```

---

Best Parameters

```python
grid.best_params_
```

Best Model

```python
grid.best_estimator_
```

---

# 10. RandomizedSearchCV

Grid Search

tries

EVERY combination.

Random Search

tries

RANDOM combinations.

Much faster.

Good for

large parameter spaces.

---

# 11. Cross Validation

Instead of

one train-test split

Cross Validation

uses

multiple splits.

Example

5-fold.

```
Fold 1

Train Train Train Train Test

Fold 2

Train Train Train Test Train

...

```

Average score

↓

More reliable.

---

Example

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(

    model,

    X,

    y,

    cv=5

)
```

Average

```python
scores.mean()
```

---

# 12. Which Should I Use?

Earlier

you asked

Cross Validation

vs

GridSearchCV.

The answer

Both.

GridSearchCV

uses

Cross Validation

internally.

Workflow

```
Cross Validation

↓

Estimate Performance

GridSearchCV

↓

Find Best Parameters

Final Model

↓

Train

↓

Test
```

---

# 13. Evaluation Metrics

Accuracy

```python
accuracy_score()
```

---

Precision

```python
precision_score()
```

Important

Spam Detection.

---

Recall

```python
recall_score()
```

How many

spam messages

did we detect?

---

F1

Best

for

imbalanced datasets.

---

Confusion Matrix

```python
confusion_matrix()
```

---

ROC Curve

```python
roc_auc_score()
```

---

Classification Report

```python
classification_report()
```

---

# 14. Your Spam Detection Project

Your confusion matrix

```
[[963   2]

 [20 130]]
```

Interpretation

963

Correct Ham

130

Correct Spam

2

False Positives

20

False Negatives

Excellent.

Notice

False Positives

are

very low.

This is desirable.

---

# 15. Probability Calibration

LinearSVC

cannot

produce

probabilities.

Only

decision scores.

If probabilities

are needed

use

```python
from sklearn.calibration import CalibratedClassifierCV
```

Example

```python
from sklearn.calibration import CalibratedClassifierCV

svm = LinearSVC()

model = CalibratedClassifierCV(svm)

model.fit(X_train,y_train)
```

Now

```python
model.predict_proba()
```

works.

---

# 16. Saving Models

```python
import joblib

joblib.dump(pipe,"spam.pkl")
```

Loading

```python
pipe = joblib.load("spam.pkl")
```

Exactly

what you used

in your Flask project.

---

# 17. Production Pipeline

Real systems

follow

```
Incoming Text

↓

Cleaning

↓

TF-IDF

↓

LinearSVC

↓

Prediction

↓

API Response

↓

Frontend
```

Never save

just

the classifier.

Always save

the entire

Pipeline.

---

# 18. Common Mistakes

❌ Forgetting scaling

---

❌ Using SVC

on

500,000 samples

---

❌ Using RBF

for TF-IDF

---

❌ Saving

only

the model

instead of

the pipeline.

---

❌ Using Accuracy

instead of

F1

for imbalanced datasets.

---

# 19. Best Practices

✔ Use Pipeline

✔ Use Cross Validation

✔ Tune C

✔ Scale numeric data

✔ Use LinearSVC for NLP

✔ Save entire Pipeline

✔ Evaluate using F1

✔ Analyze errors

---

# Interview Questions

### Q1. Why should we use a Pipeline with SVM?

A Pipeline ensures that preprocessing (such as scaling or vectorization) is applied consistently during training and prediction. It also prevents data leakage and simplifies deployment.

---

### Q2. Why is feature scaling important for SVM?

SVM relies on distances and dot products to determine the decision boundary. Features with larger numerical ranges can dominate these calculations, leading to suboptimal hyperplanes.

---

### Q3. Why is LinearSVC commonly used for text classification?

Text data transformed with TF-IDF is high-dimensional and sparse. LinearSVC is optimized for this setting, offering faster training and often equal or better performance than non-linear kernels.

---

### Q4. Why should we save the entire Pipeline instead of only the model?

The Pipeline contains both preprocessing (e.g., TF-IDF, scaling) and the trained classifier. Saving only the classifier can lead to inconsistent preprocessing and incorrect predictions in production.

---

### Q5. How does GridSearchCV differ from Cross Validation?

Cross-validation estimates model performance by evaluating it across multiple data splits. GridSearchCV uses cross-validation internally while systematically searching for the best hyperparameter combination.

---

# Cheat Sheet

- **SVC:** Supports multiple kernels; flexible but slower.
- **LinearSVC:** Linear kernel only; fast and ideal for high-dimensional sparse data.
- **NuSVC:** Alternative parameterization using ν instead of C; less commonly used.
- **StandardScaler:** Required for most numeric SVM tasks.
- **TF-IDF:** Already normalized; additional scaling is usually unnecessary.
- **Pipeline:** Keeps preprocessing and model together, preventing data leakage.
- **GridSearchCV:** Finds optimal hyperparameters using cross-validation.
- **CalibratedClassifierCV:** Adds probability estimates to classifiers like LinearSVC.
- **Production:** Save the entire pipeline with `joblib` and reuse it for inference.
